In [1]:
import os
import sys
sys.path.append('/home/royhirsch/conformal/v3')

from ml_collections import config_dict
import logging
import pandas as pd
import pickle
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
from scipy.special import softmax
import torchcp
from data_config import get_config
from conformal_modules import APS, RAPS, Naive, SAPS, platt_scale

In [2]:
config = get_config('imagenet1k_resnet50') # cifar100_resnet56, imagenet1k_resnet50
config.par_calib = 0.5
config.num_calib = int(config.num_samples * config.par_calib)
config.plat_scale = True
config.seed = 4
config.estimation_method_name = 'max' # max / mean
config.randomized = True
config.no_zero_size_sets = True
config.alpha = 0.1

# Funcs

In [3]:
with open(config.file_name, 'rb') as file:
    data = pickle.load(file)

labels_eval, labels_calib, scores_eval, scores_calib = train_test_split(
    data["labels"], data["preds"], test_size=config.par_calib, random_state=42
)
if config.plat_scale:
    t = platt_scale(scores_calib, labels_calib, max_iters=100, lr=0.01, epsilon=0.005)
else:
    t = 1.0

print(f"Temperature: {t}")
scores_calib = softmax(scores_calib / t, axis=1)
scores_eval = softmax(scores_eval / t, axis=1)
eval_data = {"labels": labels_eval, "scores": scores_eval}
calib_data = {"labels": labels_calib, "scores": scores_calib}


 10%|█         | 10/100 [00:05<00:50,  1.79it/s]


Temperature: 0.6197080612182617


# Baselines

In [4]:
n = len(calib_data["labels"])

conf_score = SAPS(randomized=config.randomized, no_zero_size_sets=config.no_zero_size_sets, seed=config.seed)
cal_scores = conf_score.get_scores(calib_data["scores"], calib_data["labels"])
baseline_qhat = np.quantile(
    cal_scores, np.ceil((n + 1) * (1 - config.alpha)) / n, interpolation="higher"
)
prediction_sets = conf_score.get_sets(eval_data["scores"], baseline_qhat)

empirical_coverage = prediction_sets[
    np.arange(prediction_sets.shape[0]), eval_data["labels"]
].mean()

baseline_mets = {
    "size": prediction_sets.sum(1).mean(),
    "coverage": empirical_coverage,
}

print('Baseline mets:')
for key, value in baseline_mets.items():
    print(f"{key}: {value:.4f}")
print('Qhat: {:.4f}'.format(baseline_qhat))


Baseline mets:
size: 1.8788
coverage: 0.8975
Qhat: 1.7217


# Ours method

In [5]:
def get_estimated_score(scores, method, conf_score):
    if method == 'max':
        scores = conf_score.get_scores(scores, scores.argmax(1))
        return scores 
    
    elif method == 'mean':
        cal_pi = scores.argsort(1)[:, ::-1]
        cal_srt = np.take_along_axis(scores, cal_pi, axis=1).cumsum(axis=1)
        cal_softmax_correct_class = np.take_along_axis(cal_srt, cal_pi.argsort(axis=1), axis=1)
        return (cal_softmax_correct_class * scores).sum(1)
    else:
        raise ValueError('Unknown method')


In [6]:
cal_scores_est = get_estimated_score(calib_data["scores"], config.estimation_method_name, conf_score)
cal_scores_true = conf_score.get_scores(calib_data["scores"], calib_data["labels"])
# cal_scores_est = np.zeros_like(cal_scores_est)
normalizer = cal_scores_true.max()
eps = 1e-10
residuales = (cal_scores_true - cal_scores_est) / (normalizer - cal_scores_est + eps)

print('Residuales: mean: {:.3f} std: {:.3f} min: {:.3f} max: {:.3f}'.format(residuales.mean(), residuales.std(), residuales.min(), residuales.max()))

qhat = np.quantile(
    residuales, np.ceil((n + 1) * (1 - config.alpha)) / n, interpolation="higher"
)

print('Qhat: {:.4f}'.format(qhat))
val_scores_est = get_estimated_score(eval_data["scores"], config.estimation_method_name, conf_score)
val_scores_est_copy = val_scores_est
val_scores_true = conf_score.get_scores(eval_data["scores"], eval_data["labels"])
# val_scores_est = np.zeros_like(val_scores_est)
val_scores_est = val_scores_est + qhat * (normalizer - val_scores_est + eps)
print('Par of scores above 1: {:.2f}'.format(np.sum(val_scores_est > 1) / len(val_scores_est)))
# val_scores_est[val_scores_est > 1] = baseline_qhat
prediction_sets = conf_score.get_sets(eval_data["scores"], val_scores_est)

empirical_coverage = prediction_sets[
    np.arange(prediction_sets.shape[0]), eval_data["labels"]
].mean()

ours_mets = {
    "size": prediction_sets.sum(1).mean(),
    "coverage": empirical_coverage,
}

print('Ours mets:')
for key, value in ours_mets.items():
    print(f"{key}: {value:.4f}")


Residuales: mean: 0.002 std: 0.025 min: -0.001 max: 1.000
Qhat: 0.0015
Par of scores above 1: 1.00
Ours mets:
size: 2.0152
coverage: 0.8989


In [7]:
# # debug #
# import matplotlib.pyplot as plt
# from sklearn.metrics import r2_score

# fig, ax = plt.subplots(1, 3, figsize=(18, 4))

# ax[0].scatter(cal_scores_true, cal_scores_est, color='blue', alpha=0.3)
# ax[0].set_title('Scatter Plot of True vs Estimated Calib Scores')
# r2 = r2_score(cal_scores_true, cal_scores_est)
# ax[0].text(0.05, 0.95, f'R² = {r2:.2f}', transform=ax[0].transAxes, 
#          fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
# plt.xlabel('True Scores')
# plt.ylabel('Estimated Scores')
# ax[0].plot([0, 1], [0, 1], 'r--')
# ax[0].grid(True)

# ax[1].scatter(val_scores_true, val_scores_est_copy, color='green', alpha=0.3)
# ax[1].set_title('Scatter Plot of True vs Estimated Eval Scores\n(without qhat correction)')
# r2 = r2_score(val_scores_true, val_scores_est_copy)
# ax[1].text(0.05, 0.95, f'R² = {r2:.2f}', transform=ax[1].transAxes, 
#          fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
# plt.xlabel('True Scores')
# plt.ylabel('Estimated Scores')
# ax[1].plot([0, 1], [0, 1], 'r--')
# ax[1].grid(True)

# ax[2].scatter(val_scores_true, val_scores_est, color='green', alpha=0.3)
# ax[2].set_title('Scatter Plot of True vs Estimated Eval Scores\n(with qhat correction)')
# r2 = r2_score(val_scores_true, val_scores_est)
# ax[2].text(0.05, 0.95, f'R² = {r2:.2f}', transform=ax[2].transAxes, 
#          fontsize=12, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))
# plt.xlabel('True Scores')
# plt.ylabel('Estimated Scores')
# ax[2].plot([0, 1], [0, 1], 'r--')
# ax[2].grid(True)
# plt.show()

# plt.figure(figsize=(10, 2))
# _ = plt.hist(val_scores_true - val_scores_est, bins=50)
# plt.title('Eval residuales')

# plt.figure(figsize=(10, 2))
# _ = plt.hist(val_scores_est, bins=50)
# plt.title('Estimated eval scores')

# Results

In [8]:
import pandas as pd

df = pd.DataFrame(columns=['Method', 'Size', 'Coverage'])
df.loc[0] = ['Baseline', baseline_mets['size'], baseline_mets['coverage']]
df.loc[1] = ['Ours', ours_mets['size'], ours_mets['coverage']]
print(df)

     Method     Size  Coverage
0  Baseline  1.87876   0.89752
1      Ours  2.01520   0.89888


In [9]:
'''
     Method    Size  Coverage
0  Baseline  6.0732    0.9278
1      Ours  4.1024    0.8986

'''

'\n     Method    Size  Coverage\n0  Baseline  6.0732    0.9278\n1      Ours  4.1024    0.8986\n\n'